In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_csv(
    "Group A Deadline-19 October 26 Model_C6 - Sheet1.csv"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Dataset shape: (259, 14)
Columns: ['S.No', 'Country', 'City', 'Company', 'LINKDLN LINK', 'Website', 'Tech Need / Problem', 'EMAIL', 'CONTACT', 'Google Map', 'Category', 'Cleaned Category', 'Rating', 'Review']


,S.No,Country,City,Company,LINKDLN LINK,Website,Tech Need / Problem,EMAIL,CONTACT,Google Map,Category,Cleaned Category,Rating,Review
0,1,India,Maharashtra,MATOSHREE MEDICAL & GENERAL STORES(मातोश्री मे...,NaN,NaN,NaN,NaN,07414919176,https://maps.app.goo.gl/DaJsWat6tMuka9rG8,Medical Shop,Pharmacy,4.9,72
1,2,India,Gujarat,Ambaji Medical Stores,NaN,NaN,NaN,NaN,09998177155,https://maps.app.goo.gl/2iUVLmXr4YPBuPcD8,Medical Shop,Pharmacy,4.6,11
2,3,India,Gujarat,Gujarat medical store,NaN,NaN,NaN,NaN,02692253927,https://maps.app.goo.gl/4ugMcf67irmfVqa97,Medical Shop,Pharmacy,4.8,10
3,4,India,Delhi,Anand Medical Stores,NaN,NaN,NaN,NaN,01126872323,https://maps.app.goo.gl/gwHi9xcA6q6wh8um8,Medical Shop,Pharmacy,3.4,8
4,5,India,Delhi,Bhardwaj Medical Store,NaN,NaN,NaN,NaN,09782527891,https://maps.app.goo.gl/yebNR3zkR6heYv5VA,Medical Shop,Pharmacy,0.0,0


In [3]:
required_columns = [
    "Company", "Country", "Rating", "Review", "Cleaned Category"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

df["Review"] = pd.to_numeric(
    df["Review"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

if df["Rating"].dropna().lt(0).any() or df["Rating"].dropna().gt(5).any():
    raise ValueError("Ratings must be between 0 and 5.")

if df["Review"].dropna().lt(0).any():
    raise ValueError("Review counts cannot be negative.")

print("Total businesses:", len(df))
print("Missing ratings:", df["Rating"].isna().sum())
print("Missing review counts:", df["Review"].isna().sum())

Total businesses: 259
Missing ratings: 0
Missing review counts: 0


In [4]:
no_data_mask = (
    df["Rating"].eq(0) &
    df["Review"].eq(0)
)

model_df = df.loc[
    ~no_data_mask &
    df["Rating"].notna() &
    df["Review"].notna() &
    df["Cleaned Category"].notna()
].copy()

print("Total businesses:", len(df))
print("Businesses with Rating=0 and Review=0:", no_data_mask.sum())
print("Businesses available for review analysis:", len(model_df))

Total businesses: 259
Businesses with Rating=0 and Review=0: 14
Businesses available for review analysis: 245


In [5]:
model_df["Log_Review_Count"] = np.log1p(model_df["Review"])

min_review = model_df["Log_Review_Count"].min()
max_review = model_df["Log_Review_Count"].max()

if max_review > min_review:
    model_df["Normalized_Review_Count"] = (
        (model_df["Log_Review_Count"] - min_review)
        / (max_review - min_review)
    )
else:
    model_df["Normalized_Review_Count"] = 0.0

model_df["Normalized_Rating"] = model_df["Rating"] / 5

model_df["Review_Performance_Score"] = (
    0.60 * model_df["Normalized_Review_Count"]
    + 0.40 * model_df["Normalized_Rating"]
)

display(
    model_df[
        [
            "Company", "Cleaned Category", "Rating", "Review",
            "Normalized_Review_Count", "Normalized_Rating",
            "Review_Performance_Score"
        ]
    ].head(10)
)

,Company,Cleaned Category,Rating,Review,Normalized_Review_Count,Normalized_Rating,Review_Performance_Score
0,MATOSHREE MEDICAL & GENERAL STORES(मातोश्री मे...,Pharmacy,4.9,72,0.429895,0.98,0.649937
1,Ambaji Medical Stores,Pharmacy,4.6,11,0.214123,0.92,0.496474
2,Gujarat medical store,Pharmacy,4.8,10,0.203725,0.96,0.506235
3,Anand Medical Stores,Pharmacy,3.4,8,0.179744,0.68,0.379846
5,Deepak Medical store,Pharmacy,4.1,169,0.530917,0.82,0.646550
6,Ram Medical Stores,Pharmacy,4.3,322,0.607622,0.86,0.708573
7,Rajat Medical Stores,Pharmacy,4.5,911,0.731666,0.90,0.799000
8,Shree dwarkadhish medical store,Pharmacy,4.0,23,0.296958,0.80,0.498175
9,METRO CITY PHARMACY,Pharmacy,3.9,48,0.382256,0.78,0.541354
10,Mangla Medicos,Pharmacy,4.2,17,0.262578,0.84,0.493547


In [6]:
# STEP 6: ONE SHARED PRIORITY PER CATEGORY

category_scores = (
    model_df.groupby("Cleaned Category")
    .agg(
        Average_Review_Score=("Review_Performance_Score", "mean"),
        Business_Count=("Company", "count")
    )
    .reset_index()
)

# Rank categories by average review score
category_scores["Category_Rank"] = (
    category_scores["Average_Review_Score"]
    .rank(method="first", ascending=False)
)

# Divide categories into three groups by rank
n_categories = len(category_scores)

category_scores["Priority"] = np.select(
    [
        category_scores["Category_Rank"] <= n_categories / 3,
        category_scores["Category_Rank"] <= 2 * n_categories / 3
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

# Apply the same category priority to every business
priority_map = category_scores.set_index(
    "Cleaned Category"
)["Priority"]

model_df["Priority"] = model_df["Cleaned Category"].map(priority_map)

print("Category counts by priority:")
print(category_scores["Priority"].value_counts())

print("\nFinal category mapping:")
display(
    category_scores.sort_values(
        "Average_Review_Score",
        ascending=False
    )
)

assert category_scores["Cleaned Category"].is_unique
assert category_scores["Priority"].isin(
    ["High", "Medium", "Low"]
).all()

Category counts by priority:
Priority
Low       6
Medium    5
High      5
Name: count, dtype: int64

Final category mapping:


,Cleaned Category,Average_Review_Score,Business_Count,Category_Rank,Priority
12,Restaurant,0.674091,29,1.0,High
10,Physiotherapy & Rehabilitation,0.667526,15,2.0,High
13,Stationery Shop,0.659792,38,3.0,High
4,Dance Academy,0.648994,6,4.0,High
2,Clinic,0.621637,46,5.0,High
0,Automotive Services,0.610940,11,6.0,Medium
7,Hospital,0.609212,9,7.0,Medium
1,Childcare,0.572765,3,8.0,Medium
6,Grocery Store,0.551975,23,9.0,Medium
9,Pharmacy,0.546642,40,10.0,Medium


In [7]:
features = [
    "Normalized_Review_Count",
    "Normalized_Rating"
]

X = model_df[features]
y = model_df["Priority"]

# Check whether a valid stratified split is possible
class_counts = y.value_counts()

if len(class_counts) < 2 or class_counts.min() < 2:
    raise ValueError(
        "Not enough examples in each priority class for a stratified "
        "train/test split. Review the category priority distribution."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, zero_division=0))

Training rows: 196
Testing rows: 49
Accuracy: 0.6327
              precision    recall  f1-score   support

        High       0.71      0.89      0.79        27
         Low       0.00      0.00      0.00         5
      Medium       0.47      0.41      0.44        17

    accuracy                           0.63        49
   macro avg       0.39      0.43      0.41        49
weighted avg       0.55      0.63      0.59        49



In [8]:
model_df["Predicted_Priority"] = model.predict(X)

probabilities = model.predict_proba(X)
model_df["Predicted_Probability"] = probabilities.max(axis=1)

display(
    model_df[
        [
            "Company", "Cleaned Category",
            "Review_Performance_Score", "Priority",
            "Predicted_Priority", "Predicted_Probability"
        ]
    ].head(10)
)

,Company,Cleaned Category,Review_Performance_Score,Priority,Predicted_Priority,Predicted_Probability
0,MATOSHREE MEDICAL & GENERAL STORES(मातोश्री मे...,Pharmacy,0.649937,Medium,High,0.656144
1,Ambaji Medical Stores,Pharmacy,0.496474,Medium,Medium,0.455619
2,Gujarat medical store,Pharmacy,0.506235,Medium,Medium,0.431753
3,Anand Medical Stores,Pharmacy,0.379846,Medium,Medium,0.603000
5,Deepak Medical store,Pharmacy,0.646550,Medium,High,0.619782
6,Ram Medical Stores,Pharmacy,0.708573,Medium,High,0.718730
7,Rajat Medical Stores,Pharmacy,0.799000,Medium,High,0.827175
8,Shree dwarkadhish medical store,Pharmacy,0.498175,Medium,Medium,0.509412
9,METRO CITY PHARMACY,Pharmacy,0.541354,Medium,Medium,0.480377
10,Mangla Medicos,Pharmacy,0.493547,Medium,Medium,0.494441


In [9]:
df["Review_Performance_Score"] = np.nan
df["Average_Review_Score"] = np.nan
df["Priority"] = "No Review Data"
df["Predicted_Priority"] = "No Review Data"
df["Predicted_Probability"] = np.nan

# Copy business-level scores and shared category priority back
for col in [
    "Review_Performance_Score",
    "Priority",
    "Predicted_Priority",
    "Predicted_Probability"
]:
    df.loc[model_df.index, col] = model_df[col]

df["Average_Review_Score"] = df["Cleaned Category"].map(
    category_scores.set_index("Cleaned Category")["Average_Review_Score"]
)

# Ensure businesses without usable review data remain labelled correctly
df.loc[no_data_mask, "Priority"] = "No Review Data"
df.loc[no_data_mask, "Predicted_Priority"] = "No Review Data"
df.loc[no_data_mask, "Predicted_Probability"] = np.nan

print("Final rows:", len(df))
print("\nShared priority counts:")
print(df["Priority"].value_counts(dropna=False))

Final rows: 259

Shared priority counts:
Priority
High              134
Medium             86
Low                25
No Review Data     14
Name: count, dtype: int64


In [10]:
for priority in ["High", "Medium", "Low"]:
    priority_df = df[df["Priority"] == priority].copy()

    priority_df.to_csv(
        f"BLC_C6_{priority}_Priority.csv",
        index=False
    )

    print(f"{priority}: {len(priority_df)} businesses exported")

df[df["Priority"] == "No Review Data"].to_csv(
    "BLC_C6_No_Review_Data.csv",
    index=False
)

High: 134 businesses exported
Medium: 86 businesses exported
Low: 25 businesses exported


In [11]:
category_scores.to_csv(
    "BLC_C6_Category_Priority_Mapping.csv",
    index=False
)

for priority in ["High", "Medium", "Low"]:
    categories = sorted(
        category_scores.loc[
            category_scores["Priority"] == priority,
            "Cleaned Category"
        ].dropna().unique()
    )

    print(f"\n{priority} Priority:")
    print(", ".join(categories) if categories else "None")


High Priority:
Clinic, Dance Academy, Physiotherapy & Rehabilitation, Restaurant, Stationery Shop

Medium Priority:
Automotive Services, Childcare, Grocery Store, Hospital, Pharmacy

Low Priority:
Computer Support and Service, Education & Tutoring, Music School, Plant Nursery, Supermarket, Training School


In [12]:
country_clean = df["Country"].astype("string").str.strip().str.casefold()

df["Business_Classification"] = np.select(
    [
        country_clean.eq("india"),
        country_clean.notna() & country_clean.ne("")
    ],
    [
        "Indian",
        "Foreign"
    ],
    default="Unknown"
)

indian_df = df[df["Business_Classification"] == "Indian"].copy()
foreign_df = df[df["Business_Classification"] == "Foreign"].copy()
unknown_df = df[df["Business_Classification"] == "Unknown"].copy()

indian_df.to_csv("BLC_C6_Indian_Businesses.csv", index=False)
foreign_df.to_csv("BLC_C6_Foreign_Businesses.csv", index=False)
unknown_df.to_csv("BLC_C6_Unknown_Country.csv", index=False)

print("Indian:", len(indian_df))
print("Foreign:", len(foreign_df))
print("Unknown country:", len(unknown_df))

Indian: 200
Foreign: 59
Unknown country: 0


In [13]:
assert category_scores["Cleaned Category"].is_unique

for filename in [
    "BLC_C6_High_Priority.csv",
    "BLC_C6_Medium_Priority.csv",
    "BLC_C6_Low_Priority.csv",
    "BLC_C6_No_Review_Data.csv",
    "BLC_C6_Category_Priority_Mapping.csv",
    "BLC_C6_Indian_Businesses.csv",
    "BLC_C6_Foreign_Businesses.csv",
    "BLC_C6_Unknown_Country.csv"
]:
    check_df = pd.read_csv(filename)
    print(f"{filename}: {len(check_df)} rows")

print("\nAll exports created.")

BLC_C6_High_Priority.csv: 134 rows
BLC_C6_Medium_Priority.csv: 86 rows
BLC_C6_Low_Priority.csv: 25 rows
BLC_C6_No_Review_Data.csv: 14 rows
BLC_C6_Category_Priority_Mapping.csv: 16 rows
BLC_C6_Indian_Businesses.csv: 200 rows
BLC_C6_Foreign_Businesses.csv: 59 rows
BLC_C6_Unknown_Country.csv: 0 rows

All exports created.


In [14]:
display(
    category_scores[
        ["Cleaned Category", "Average_Review_Score", "Business_Count", "Priority"]
    ].sort_values("Average_Review_Score", ascending=False)
)

,Cleaned Category,Average_Review_Score,Business_Count,Priority
12,Restaurant,0.674091,29,High
10,Physiotherapy & Rehabilitation,0.667526,15,High
13,Stationery Shop,0.659792,38,High
4,Dance Academy,0.648994,6,High
2,Clinic,0.621637,46,High
0,Automotive Services,0.610940,11,Medium
7,Hospital,0.609212,9,Medium
1,Childcare,0.572765,3,Medium
6,Grocery Store,0.551975,23,Medium
9,Pharmacy,0.546642,40,Medium


In [15]:
for priority in ["High", "Medium", "Low"]:
    priority_df = df[df["Priority"] == priority].copy()

    filename = f"BLC_C6_{priority}_Priority.csv"
    priority_df.to_csv(filename, index=False)

    print(f"{filename}: {len(priority_df)} businesses")

BLC_C6_High_Priority.csv: 134 businesses
BLC_C6_Medium_Priority.csv: 86 businesses
BLC_C6_Low_Priority.csv: 25 businesses
